## FINE-TUNING A LARGE LANGUAGE MODEL (LLM) FOR TEXT CLASSIFICATION


### FOCUS : 
LARGE LANGUAGE MODEL (LLM), TRANSFER LEARNING and NLP

### AIM :
Utilize transfer learning to fine-tune a pre-trained small Large Language Model(LLM) (like DistilBERT) for a specific NLP task, such as sentiment analysis (binary classification).

### INSTRUCTIONS :
1. Install the Hugging Face transformers library, datasets, and PyTorch/TensorFlow

2. Data Acquisition 

3. Model & Tokenizer Loading 

4. Data Prepocessing 

5. Fine-Tuning 

6. Evaluation and Inference

### STEPS : 
1. Installing Libraries
2. Importing Libraries
3. Downloading the IMDb Dataset 
4. Loading the tokenizer
5. Creating our Dataset
6. Tokenizing the training data 
7. Loading the model 
8. Training the model 
9. Testing the dataset
10. Tokenizing the test data 
11. Evaluating the model 
12. Defining the Prediction function 
13. Example

## 1. Installing Libraries

In [1]:
%pip install -U transformers evaluate accelerate torch scikit-learn requests pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Importing Libraries

In [ ]:
import requests
import pandas as pd
import torch
import time
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Libraries imported successfully!")

Libraries imported successfully!


## 3. Downloading the IMDB dataset

In [3]:
def get_imdb_data(split, offset, total_rows):
    rows = []

    for current_offset in range(offset, offset + total_rows, 100):
        length = min(100, offset + total_rows - current_offset)

        params = {
            "dataset": "stanfordnlp/imdb",
            "config": "plain_text",
            "split": split,
            "offset": current_offset,
            "length": length
        }

        for attempt in range(5):
            try:
                response = requests.get(
                    "https://datasets-server.huggingface.co/rows",
                    params=params,
                    timeout=90
                )

                if response.status_code == 200:
                    batch = response.json()["rows"]
                    rows.extend([item["row"] for item in batch])
                    print(
                        f"{split}: {len(rows)}/{total_rows} "
                        f"(offset {current_offset})"
                    )
                    break

                print(f"Attempt {attempt + 1}: HTTP {response.status_code}")

            except requests.RequestException as e:
                print(f"Attempt {attempt + 1}: {e}")

            time.sleep(3)

        else:
            raise Exception(
                f"Failed to download batch at offset {current_offset}"
            )

    return pd.DataFrame(rows)


negative_train = get_imdb_data("train", 0, 500)
positive_train = get_imdb_data("train", 12500, 500)

balanced_train_df = pd.concat(
    [negative_train, positive_train],
    ignore_index=True
)

print("\nTraining shape:", balanced_train_df.shape)
print("\nClass distribution:")
print(balanced_train_df["label"].value_counts())

train: 100/500 (offset 0)
train: 200/500 (offset 100)
train: 300/500 (offset 200)
train: 400/500 (offset 300)
train: 500/500 (offset 400)
train: 100/500 (offset 12500)
train: 200/500 (offset 12600)
train: 300/500 (offset 12700)
train: 400/500 (offset 12800)
train: 500/500 (offset 12900)

Training shape: (1000, 2)

Class distribution:
label
0    500
1    500
Name: count, dtype: int64


In [4]:
print("Number of training reviews:", len(balanced_train_df))

print("\nFirst review:")
print(balanced_train_df.iloc[0]["text"])

print("\nFirst review label:")
print(balanced_train_df.iloc[0]["label"])

Number of training reviews: 1000

First review:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the se

## 4. Loading the tokenizer

In [5]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


## 5. Creating our Dataset


In [6]:
class IMDbDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

print("Dataset class created successfully!")

Dataset class created successfully!


## 6. Tokenizing the training data

In [7]:
balanced_train_encodings = tokenizer(
    balanced_train_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

balanced_train_dataset = IMDbDataset(
    balanced_train_encodings,
    balanced_train_df["label"].tolist()
)

balanced_train_loader = DataLoader(
    balanced_train_dataset,
    batch_size=8,
    shuffle=True
)

print("Training samples:", len(balanced_train_dataset))
print("Training batches:", len(balanced_train_loader))

Training samples: 1000
Training batches: 125


## 7. Loading the Model


In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

print("Using device:", device)
print("Model loaded successfully!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using device: cpu
Model loaded successfully!


## 8. Training the model

In [9]:
model.train()

for epoch in range(2):

    total_loss = 0

    for step, batch in enumerate(balanced_train_loader):

        inputs = {
            "input_ids": batch["input_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device),
            "labels": batch["labels"].to(device)
        }

        optimizer.zero_grad()

        outputs = model(**inputs)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        if (step + 1) % 25 == 0:
            print(
                f"Epoch {epoch + 1}/2 | "
                f"Step {step + 1}/{len(balanced_train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    average_loss = total_loss / len(balanced_train_loader)

    print(
        f"Epoch {epoch + 1} completed | "
        f"Average Loss: {average_loss:.4f}"
    )

print("Training completed!")

Epoch 1/2 | Step 25/125 | Loss: 0.7638
Epoch 1/2 | Step 50/125 | Loss: 0.4298
Epoch 1/2 | Step 75/125 | Loss: 0.5362
Epoch 1/2 | Step 100/125 | Loss: 0.4833
Epoch 1/2 | Step 125/125 | Loss: 0.8240
Epoch 1 completed | Average Loss: 0.5401
Epoch 2/2 | Step 25/125 | Loss: 0.1819
Epoch 2/2 | Step 50/125 | Loss: 0.0945
Epoch 2/2 | Step 75/125 | Loss: 0.2502
Epoch 2/2 | Step 100/125 | Loss: 0.2970
Epoch 2/2 | Step 125/125 | Loss: 0.0595
Epoch 2 completed | Average Loss: 0.2661
Training completed!


## 9. Testing the Dataset

In [10]:
# Load 250 negative and 250 positive test reviews

negative_test = get_imdb_data("test", 0, 250)
positive_test = get_imdb_data("test", 12500, 250)

balanced_test_df = pd.concat(
    [
        negative_test,
        positive_test
    ],
    ignore_index=True
)

print("\nTesting shape:", balanced_test_df.shape)
print("\nClass distribution:")
print(balanced_test_df["label"].value_counts())

test: 100/250 (offset 0)
test: 200/250 (offset 100)
test: 250/250 (offset 200)
test: 100/250 (offset 12500)
test: 200/250 (offset 12600)
test: 250/250 (offset 12700)

Testing shape: (500, 2)

Class distribution:
label
0    250
1    250
Name: count, dtype: int64


## 10. Tokenizing the test data

In [11]:
balanced_test_encodings = tokenizer(
    balanced_test_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

balanced_test_dataset = IMDbDataset(
    balanced_test_encodings,
    balanced_test_df["label"].tolist()
)

balanced_test_loader = DataLoader(
    balanced_test_dataset,
    batch_size=8,
    shuffle=False
)

print("Testing samples:", len(balanced_test_dataset))
print("Testing batches:", len(balanced_test_loader))

Testing samples: 500
Testing batches: 63


## 11. Evaluating the model 

In [12]:
from sklearn.metrics import accuracy_score, f1_score

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for batch in balanced_test_loader:

        inputs = {
            "input_ids": batch["input_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device),
            "labels": batch["labels"].to(device)
        }

        outputs = model(**inputs)

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_preds.extend(predictions.cpu().tolist())
        all_labels.extend(inputs["labels"].cpu().tolist())

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"F1 Score: {f1:.4f}")

Accuracy: 0.8280 (82.80%)
F1 Score: 0.8333


In [13]:
model.save_pretrained("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")

print("Model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!


## 12. Defining the Prediction Function

In [14]:
def predict_sentiment(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
        if key in ["input_ids", "attention_mask"]
    }

    model.eval()

    with torch.no_grad():

        outputs = model(**inputs)

        prediction = torch.argmax(
            outputs.logits,
            dim=1
        ).item()

    return "Positive" if prediction == 1 else "Negative" 

## 13. Example 

In [17]:
review1 = "This movie was absolutely great. I loved every minute of it!"
review2 = "This movie was terrible and extremely boring. I hated it."

print("Review 1:", predict_sentiment(review1))
print("Review 2:", predict_sentiment(review2))

Review 1: Positive
Review 2: Negative
